In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
f = pd.read_csv("Mall_Customers.csv" )
f = f.dropna()
f["GenderNo"] = np.where(f["Gender"]== "Male" , 1 ,3 )
df = f[['GenderNo', "Annual_Income", "Spending_Score"]]
scaler = StandardScaler()
dfnew = scaler.fit_transform(df)
# decide k and run kmeans
scores = {}
for k in range(2,11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(dfnew)
    scores[k] = silhouette_score(dfnew, km.labels_)
best_k = max(scores, key=scores.get)
km = KMeans(n_clusters=best_k, n_init=10, random_state=42)
labels = km.fit_predict(dfnew)
spend_mean = df["Spending_Score"].mean()
income_mean = df["Annual_Income"].mean()
def label(row):
    hi_spend = row["Spending_Score"] > spend_mean
    low_spend = row["Spending_Score"] < spend_mean
    hi_salary = row["Annual_Income"] > income_mean
    low_salary = row["Annual_Income"] < income_mean
    Male  = row["GenderNo"] < 2
    Female  = row["GenderNo"] > 2
    if hi_spend and low_salary and Male:
        return "Male With High Spend and Low Income"
    if hi_spend and hi_salary and Male:
        return "Male With High Spend and High Income"
    if hi_spend and low_salary and Female:
        return "Female With High Spend and Low Income"
    if hi_spend and hi_salary and Female:
        return "Female With High Spend and High Income"
    if low_spend and low_salary and Male:
        return "Male With Low Spend and Low Income"
    if low_spend and hi_salary and Male:
        return "Male With Low Spend and High Income"
    if low_spend and low_salary and Female:
        return "Female With Low Spend and Low Income"
    if low_spend and hi_salary and Female:
        return "Female With Low Spend and High Income"
df['Label_Kmeans'] = labels
# eps could be 0.4 or 0.5
db = DBSCAN(eps=0.4, min_samples=5)
label1 = db.fit_predict(dfnew)
df["Age"] = f["Age"]
df["Gender"] = f["Gender"]
df["CustomerID"] = f["CustomerID"]
df["Label_DBSCAN"] = label1
df['About'] = df.apply(label, axis=1)
df.to_csv("new.csv")
def recmendation_by_text(text):
    rcc= []
    if text.empty:
        print("Please give a valid string and Non empty string")
        return 
    if "male" in text.lower():
        x = df[df["Gender"] == "Male"]["CustomerID"]
        x = x.tolist()
        rcc.extend(x)
    elif "female" in text.lower():
        x = df[df["Gender"] == "Female"]["CustomerID"]
        x = x.tolist()
        rcc.extend(x)
    elif "high" in text.lower():
        if "income" in text.lower():
            x = df[df["Annual_Income"] > income_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
        if "spend" in text.lower():
            x = df[df["Spending_Score"] > spend_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
    elif "low" in text.lower():
        if "income" in text.lower():
            x = df[df["Annual_Income"] < income_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
        if "spend" in text.lower():
            x = df[df["Spending_Score"] < spend_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
    return rcc
# recmendation_by_text("high income")
def recomendation_by_person(ID: any):
    about_values = df.loc[df["CustomerID"] == ID, "About"]
    if about_values.empty:
        print(f"No record found for CustomerID {ID}")
        return []
    about_value = about_values.iloc[0]
    matching_ids = df.loc[df["About"] == about_value, "CustomerID"]
    print(f"{matching_ids.tolist()} ")
    return matching_ids.tolist()
# recomendation_by_person(10)

C:\Users\faiza\AppData\Local\Temp\ipykernel_20396\559054391.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label_Kmeans'] = labels
C:\Users\faiza\AppData\Local\Temp\ipykernel_20396\559054391.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Age"] = f["Age"]


In [8]:
def plot_2d_matplotlib(df, label_col, fname):
    plt.figure(figsize=(8, 6))
    labels = df[label_col].unique()
    for lab in sorted(labels, key=str):
        sub = df[df[label_col] == lab]
        plt.scatter(sub["Annual_Income"], sub["Spending_Score"], label=str(lab), s=40, alpha=0.75)
    plt.xlabel("Annual Income")
    plt.ylabel("Spending Score")
    plt.title(f"Customer Clusters ({label_col})")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()
def plot_3d_matplotlib(df, label_col, fname):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    labels = sorted(df[label_col].unique(), key=str)
    for lab in labels:
        sub = df[df[label_col] == lab]
        ax.scatter(sub["Age"], sub["Annual_Income"], sub["Spending_Score"], label=str(lab), s=30, alpha=0.75)
    ax.set_xlabel("Age")
    ax.set_ylabel("Annual Income (k$)")
    ax.set_zlabel("Spending Score")
    ax.set_title(f"3D Customer Clusters ({label_col})")
    ax.legend(loc="best", fontsize=7)
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close()
output = "output_plot"
plot_2d_matplotlib(df,"Label_Kmeans", f"{output}/kmeans_2d.png")
plot_3d_matplotlib(df,"Label_Kmeans", f"{output}/kmeans_3d.png")
plot_2d_matplotlib(df,"Label_DBSCAN", f"{output}/DBSCAN_2d.png")
plot_3d_matplotlib(df,"Label_DBSCAN", f"{output}/DBSCAN_3d.png")